In [5]:
import pandas as pd
df = pd.read_csv('/Users/noahmancione/code projects/samhsa_data_analysis/data/tedsd_puf_2023.csv')
print(df.shape)
print(df.dtypes)
df.head()
#check reason column
print(df['REASON'].value_counts().sort_index())


(1474025, 76)
DISYR       int64
CASEID      int64
STFIPS      int64
EDUC        int64
MARSTAT     int64
            ...  
DIVISION    int64
REGION      int64
IDU         int64
ALCDRUG     int64
CBSA2020    int64
Length: 76, dtype: object
REASON
1    627897
2    324461
3     60662
4    370294
5     16009
6      3730
7     70972
Name: count, dtype: int64


In [6]:
# Create binary completion column
df['completed'] = (df['REASON'] == 1).astype(int)

# Verify it worked
print(df['completed'].value_counts())
print(f"\nOverall completion rate: {df['completed'].mean():.1%}")

completed
0    846128
1    627897
Name: count, dtype: int64

Overall completion rate: 42.6%


In [7]:
import numpy as np

# Replace all -9 missing value codes with NaN
df.replace(-9, np.nan, inplace=True)

# Check how many missing values we now have per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

print(missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False))

                       missing_count  missing_pct
DETCRIM                      1268243         86.0
FREQ3_D                      1253528         85.0
FRSTUSE3                     1211094         82.2
ROUTE3                       1206565         81.9
FREQ3                        1205917         81.8
DETNLF_D                     1132882         76.9
DETNLF                       1127855         76.5
FREQ2_D                      1089097         73.9
PREG                          993700         67.4
DAYWAIT                       864212         58.6
PRIMPAY                       832872         56.5
FRSTUSE2                      791516         53.7
ROUTE2                        776162         52.7
FREQ2                         777249         52.7
HLTHINS                       679176         46.1
PRIMINC                       566916         38.5
CBSA2020                      493180         33.5
FREQ1_D                       464979         31.5
LIVARAG_D                     365392         24.8


In [8]:
# Select only the columns we need
cols = ['DISYR', 'REASON', 'completed', 'SUB1', 
        'AGE', 'SEX', 'RACE', 'LOS', 'SERVICES']

df_clean = df[cols].copy()

print(df_clean.shape)
print(df_clean.isnull().sum())
df_clean.head()

(1474025, 9)
DISYR             0
REASON            0
completed         0
SUB1         148392
AGE               0
SEX             909
RACE          70421
LOS               0
SERVICES          0
dtype: int64


,DISYR,REASON,completed,SUB1,AGE,SEX,RACE,LOS,SERVICES
0,2023,1,1,2.0,7,2.0,1.0,33,7
1,2023,1,1,2.0,9,2.0,1.0,35,7
2,2023,3,0,2.0,6,2.0,1.0,37,7
3,2023,3,0,2.0,11,2.0,1.0,36,7
4,2023,3,0,2.0,11,2.0,1.0,31,7


In [10]:
# Define mappings from codebook
sub1_map = {
    1: 'None', 2: 'Alcohol', 3: 'Cocaine/Crack',
    4: 'Marijuana', 5: 'Heroin', 6: 'Methadone',
    7: 'Other Opiates', 8: 'PCP', 9: 'Hallucinogens',
    10: 'Methamphetamine', 11: 'Other Amphetamines',
    12: 'Other Stimulants', 13: 'Benzodiazepines',
    14: 'Other Tranquilizers', 15: 'Barbiturates',
    16: 'Other Sedatives', 17: 'Inhalants',
    18: 'OTC Medications', 19: 'Other Drugs'
}

age_map = {
    1: '12-14', 2: '15-17', 3: '18-20', 4: '21-24',
    5: '25-29', 6: '30-34', 7: '35-39', 8: '40-44',
    9: '45-49', 10: '50-54', 11: '55-64', 12: '65+'
}

sex_map = {1: 'Male', 2: 'Female'}

race_map = {
    1: 'Alaska Native', 2: 'American Indian',
    3: 'Asian or Pacific Islander', 4: 'Black or African American',
    5: 'White', 6: 'Asian', 7: 'Other',
    8: 'Two or More Races', 9: 'Native Hawaiian or Pacific Islander'
}

# Apply mappings
df_clean['SUB1'] = df_clean['SUB1'].map(sub1_map)
df_clean['AGE'] = df_clean['AGE'].map(age_map)
df_clean['SEX'] = df_clean['SEX'].map(sex_map)
df_clean['RACE'] = df_clean['RACE'].map(race_map)

# Confirm it worked
df_clean.head()

,DISYR,REASON,completed,SUB1,AGE,SEX,RACE,LOS,SERVICES
0,2023,1,1,Alcohol,35-39,Female,Alaska Native,33,7
1,2023,1,1,Alcohol,45-49,Female,Alaska Native,35,7
2,2023,3,0,Alcohol,30-34,Female,Alaska Native,37,7
3,2023,3,0,Alcohol,55-64,Female,Alaska Native,36,7
4,2023,3,0,Alcohol,55-64,Female,Alaska Native,31,7


In [11]:
df_clean.to_csv('../data/tedsd_clean.csv', index=False)
print("Clean file saved successfully")
print(f"Final shape: {df_clean.shape}")

Clean file saved successfully
Final shape: (1474025, 9)


# 01 - Data Cleaning

This notebook loads the raw SAMHSA TEDS-D 2023 dataset (1,474,025 discharge records), 
replaces SAMHSA missing value codes (-9) with NaN, selects the 9 variables relevant 
to this analysis, decodes numeric codes into readable labels using the official codebook, 
and exports a cleaned dataset for analysis.

Key decisions:
- Columns with >50% missing data were excluded from analysis
- SUB1 (10.1% missing) retained but missing rows excluded during substance-level analysis
- REASON binarized into a completion indicator (1 = completed, 0 = all other outcomes)